In [220]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import shap
import os
import unicodedata

#Revisión de los archivos originales para ver cuáles columnas tiene cada año y poder detectar si cambiaron los nombres de las categorías pre unificación

archivos_excel = [
    'DATOS/Resultados de la votación 2017.xlsx',
    'DATOS/Resultados de la votación 2018.xlsx',
    'DATOS/Resultados de la votación 2019.xlsx',
    'DATOS/Resultados de la votación 2021.xlsx',
    'DATOS/Resultados de la votación 2022.xlsx',
    'DATOS/Resultados de la votación 2023.xlsx',
    'DATOS/Resultados de la votación 2024.xlsx',
    'DATOS/Resultados de la votación 2025.xls'
]

for nombre in archivos_excel:
    temp_df = pd.read_excel(nombre)
    print(f"--- Columnas en {nombre} ---")
    print(temp_df.columns.tolist())
    print("\n")

--- Columnas en DATOS/Resultados de la votación 2017.xlsx ---
['Cantidad de votos', 'Orden en la votación', 'GANADOR', 'BARRIO', 'PROYECTO', 'PRESUPUESTO ESTIMADO', 'Suma presupuesto proyectos ganadores']


--- Columnas en DATOS/Resultados de la votación 2018.xlsx ---
['Unnamed: 0', 'Olivos - 5 proyectos ganadores', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']


--- Columnas en DATOS/Resultados de la votación 2019.xlsx ---
['Nº DE PROYECTO EN LA BOLETA DEL BARRIO', 'Cantidad de votos obtenidos por el proyecto', 'Orden en la votación seguún cantidad de votos obtenidos', 'GANADOR', 'BARRIO', 'PROYECTO', 'PRESUPUESTO ESTIMADO', 'Observaciones', 'Suma presupuestos proyectos ganadores por barrio']


--- Columnas en DATOS/Resultados de la votación 2021.xlsx ---
['Nro. Proyecto en el Barrio', 'Cantidad de votos obtenidos', 'Orden en la votación según cantidad de votos', 'GANADOR', 'Barrio', 'Proyecto', 'Descripción', 'Presupuesto Estimado', 'Observaciones', 'Suma presupuestos proyectos ganadores

c:\Users\cerru\OneDrive\Documentos\TP FINAL DATOS\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


--- Columnas en DATOS/Resultados de la votación 2022.xlsx ---
['N° Proyecto en el Barrio', 'Cantidad de votos obtenidos', 'Orden en la votación según cantidad de votos', 'GANADOR', 'Barrio', 'Proyecto', 'Descripción', 'Presupuesto Estimado', 'Observaciones', 'Suma presupuestos proyectos ganadores por barrio', 'Disponible para el barrio en 2023']


--- Columnas en DATOS/Resultados de la votación 2023.xlsx ---
['N° Proy en el Barrio', 'Cantidad de votos obtenidos', 'Orden en la votación', 'GANADOR', 'Num barrio', 'Barrio', 'Proyecto', 'Descripción', 'Presupuesto Estimado', 'Observaciones', 'Disponible para el barrio en 2024']


--- Columnas en DATOS/Resultados de la votación 2024.xlsx ---
['N° Proy en el Barrio', 'Cantidad de votos obtenidos', 'Orden en la votación', 'GANADOR', 'Num barrio', 'Barrio', 'Proyecto', 'Descripción', 'Presupuesto Estimado', 'Observaciones', 'Disponible para el barrio en 2025']


--- Columnas en DATOS/Resultados de la votación 2025.xls ---
['N° Proy en el Barri

c:\Users\cerru\OneDrive\Documentos\TP FINAL DATOS\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [221]:
#El archivo de 2018 tenía los datos separados por hoja por barrio. Se recorre cada hoja, se le asigna el nombre del barrio a cada fila y luego las une en una sola tabla para analizar el año completo

archivo_2018 = 'DATOS/Resultados de la votación 2018.xlsx'

hojas_dict = pd.read_excel(archivo_2018, sheet_name=None, header=1)

lista_de_tablas = []

for nombre_de_la_hoja, tabla_de_la_hoja in hojas_dict.items():
    tabla_de_la_hoja.columns = [str(c).lower().strip() for c in tabla_de_la_hoja.columns]
    
    tabla_de_la_hoja['barrio'] = nombre_de_la_hoja
    
    lista_de_tablas.append(tabla_de_la_hoja)

df_2018_completo = pd.concat(lista_de_tablas, ignore_index=True)

print("Barrios procesados de las hojas:")
print(df_2018_completo['barrio'].unique())

df_2018_completo.head()

Barrios procesados de las hojas:
<StringArray>
[        'Olivos', 'Villa Martelli',  'Florida Oeste',          'Munro',
     'Carapachay',  'Villa Adelina',      'La Lucila',   'Florida Este',
  'Vicente López']
Length: 9, dtype: str


,número de proyecto,descripción,votos válidos,presupuesto estimado del proyecto,observaciones,barrio,votos
0,NaN,NaN,NaN,NaN,PP del barrio: $ 9.833.333,Olivos,NaN
1,9.0,ASOCIACIÓN VECINAL DE FOMENTO Y CULTURA NORTE ...,2753.0,7000000.0,GANADOR,Olivos,NaN
2,27.0,ESCUELAS PRIMARIA Y SECUNDARIA Nº 1: equipamiento,1590.0,2000000.0,GANADOR,Olivos,NaN
3,5.0,ESCUELA PRIMARIA Nº 2: reparaciones varias en ...,1498.0,2000000.0,excede pres. Esc. Provinciales,Olivos,NaN
4,10.0,COLEGIO VIRGEN DEL CARMEN:Cambiar cielorrasos ...,1323.0,438800.0,GANADOR,Olivos,NaN


In [222]:
#Se mapea y limpian los datos de 2018, renombrando columnas originales a nombres comunes con los otros años. 
#Además filtra la info relevante y elimina las filas vacías
dict_hojas = pd.read_excel('DATOS/Resultados de la votación 2018.xlsx', sheet_name=None, header=1)

lista_final_2018 = []

for nombre_barrio, df_hoja in dict_hojas.items():
    df_hoja.columns = [str(c).lower().strip() for c in df_hoja.columns]
    
    mapeo = {
        'votos válidos': 'votos',
        'descripción': 'proyecto',
        'presupuesto estimado del proyecto': 'presupuesto',
        'observaciones': 'ganador'
    }
    
    df_hoja = df_hoja.rename(columns=mapeo)
    
    df_hoja['barrio'] = nombre_barrio
    df_hoja['año'] = 2018
    
    cols_interes = ['votos', 'barrio', 'proyecto', 'presupuesto', 'ganador', 'año']
    df_hoja = df_hoja[[c for c in cols_interes if c in df_hoja.columns]]
    
    lista_final_2018.append(df_hoja)

df_2018_final = pd.concat(lista_final_2018, ignore_index=True)

df_2018_final = df_2018_final.dropna(subset=['votos', 'proyecto'], how='all')

print("Ahora todos los barrios están unificados.")
df_2018_final.sample(10)

Ahora todos los barrios están unificados.


,votos,barrio,proyecto,presupuesto,ganador,año
5,1122.0,Olivos,Escuela Municipal Paula Albarracín: aires acon...,897000.0,excede presupuesto,2018
99,437.0,Munro,Jardín de Infantes Nº 910: Juegos infantiles y...,810000.0,excede presupuesto,2018
40,526.0,Villa Martelli,CLUB LAPRIDA: Cubierta sobre losa,6300000.0,excede presupuesto,2018
3,1498.0,Olivos,ESCUELA PRIMARIA Nº 2: reparaciones varias en ...,2000000.0,excede pres. Esc. Provinciales,2018
193,1525.0,Vicente López,ISFD Nº 39: Nva. etapa para el Centro Cultura...,1500000.0,PROYECTO GANADOR,2018
131,147.0,Carapachay,JUNTA VECINAL MANUEL BELGRANO: construcción de...,980000.0,excede presupuesto,2018
26,259.0,Olivos,LA ECOLOGIA COMIENZA POR CASA: talleres en jar...,310000.0,excede presupuesto,2018
176,418.0,Florida Este,JARDÍN MATERNAL MUNICIPAL N° 10: cerramiento m...,900000.0,excede presupuesto,2018
147,36.0,Villa Adelina,PLAZA JOSÉ INGENIEROS: pista para patín,1750000.0,excede presupuesto,2018
25,291.0,Olivos,CESTOS PAPELEROS EN EL BARRIO,486000.0,excede presupuesto,2018


In [223]:
#Se clasifican en temáticas los proyectos de acuerdo a palabras clave que identifican cada uno
def categorizar_proyecto(texto):
    # Pasamos a minúsculas y eliminamos acentos del texto de entrada
    texto = str(texto).lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    
    # Categoría: Educación
    if any(w in texto for w in ['escuela', 'jardin', 'colegio', 'secundaria', 'primaria', 'estudiantil', 'educacion', 'lectura', 'aula', 'pizarron', 'escuelita', 'educativo','docente','instituto','infantil','formacion']):
        return 'Educación'
    
    # Categoría: Seguridad
    elif any(w in texto for w in ['camara', 'seguridad', 'patrullero', 'iluminacion', 'luces', 'alarma', 'totem', 'vigilancia', 'punto seguro']):
        return 'Seguridad'
    
    # Categoría: Salud
    elif any(w in texto for w in ['salud', 'hospital', 'unidad de respuesta', 'salita', 'maternal', 'ambulatoria', 'vacunatorio', 'medico', 'ambulancia', 'emergencia', 'emergencias','droga','sangre']):
        return 'Salud'
    
    # Categoría: Deporte
    elif any(w in texto for w in ['deporte', 'club', 'gimnasio', 'playon', 'cancha', 'polideportivo', 'entrenamiento']):
        return 'Deporte'
    
    # Categoría: Cultura
    elif any(w in texto for w in ['cultura', 'teatro', 'museo', 'centro cultural', 'biblioteca', 'talleres', 'musica', 'danza', 'orquesta', 'turismo','cine']):
        return 'Cultura'

    # Categoría: Medio Ambiente
    elif any(w in texto for w in ['reciclaje', 'ecologia', 'ecosistema', 'ecologica','arboles', 'huerta', 'puntos verdes', 'sustentable', 'compost', 'separacion de residuos', 'ecologico','punto verde', 'plantado', 'cesto', 'cestos', 'reciclado', 'basura', 'tierra', 'contenedor', 'ambiental','botella','bolsa']):
        return 'Medio Ambiente'

    # Categoría: Movilidad
    elif any(w in texto for w in ['semaforo', 'señalizacion','transporte', 'bicisenda', 'ciclovia', 'parada de colectivo', 'refugio', 'transito', 'estacionamiento', 'reductores', 'reducidores', 'carteles', 'carteleria','sendas peatonales','sendas','velocidad','peatonales','vial','senal','metrobus']):
        return 'Movilidad'

    # Categoría: Espacio Público
    elif any(w in texto for w in ['plaza', 'parque','arbolar', 'arbolado','canil', 'vereda', 'bacheo', 'espacio publico', 'espacios publicos', 'plazoleta', 'asfalto', 'pavimento', 'luminaria led','limpio','bebederos']):
        return 'Espacio Público'

    # Categoría: Social
    elif any(w in texto for w in ['tercera edad', 'juventud', 'adultos mayores', 'genero', 'discapacidad', 'integracion', 'comedor', 'merendero', 'social', 'jubilados', 'scout', 'centro comunitario', 'sociedad', 'fomento' , 'hogar', 'asociacion', 'fundacion', 'circulo', 'union','recreativo','lactancia']):
        return 'Social'

    # Categoría: Reparaciones/Obra
    elif any(w in texto for w in ['pintura','rampa', 'rampas','construccion', 'reconstruccion', 'instalacion', 'remodelacion', 'mejoramiento edilicio','refaccion', 'obra', 'obras', 'ampliacion', 'cambio','luminarias','edilicias','edilicia','techo','reparacion','iluminando']):
        return 'Reparaciones/Obra'
    
    #Categoría: Bomberos
    elif any(w in texto for w in ['bombero','voluntario']):
        return 'Bomberos Voluntarios'
    
    #Categoría: Malvinas
    elif any(w in texto for w in['malvinas','veteranos']):
        return 'Malvinas'
    
    elif any(w in texto for w in ['iglesia','parroquia','virgen','capilla']):
        return 'Instituciones religiosas'
    
    elif any(w in texto for w in['cruz','roja']):
        return 'Cruz Roja Argentina'

    else:
        return 'Otros/Varios'

In [224]:
#Se procesan los años estandarizando los nombres de las columnas y chequear que todos los archivos se carguen sin errores de compatibilidad
archivos_otros_años = {
    2017: 'DATOS/Resultados de la votación 2017.xlsx',
    2019: 'DATOS/Resultados de la votación 2019.xlsx',
    2021: 'DATOS/Resultados de la votación 2021.xlsx',
    2022: 'DATOS/Resultados de la votación 2022.xlsx',
    2023: 'DATOS/Resultados de la votación 2023.xlsx',
    2024: 'DATOS/Resultados de la votación 2024.xlsx',
    2025: 'DATOS/Resultados de la votación 2025.xls'
}

lista_dfs = []
for año, archivo in archivos_otros_años.items():
    engine = 'xlrd' if archivo.endswith('.xls') else 'openpyxl'
    
    df_temp = pd.read_excel(archivo, engine=engine)
    
    df_temp.columns = [str(c).lower().strip() for c in df_temp.columns]
    
    mapeo = {
        'cantidad de votos obtenidos': 'votos', 
        'cantidad de votos': 'votos', 
        'presupuesto estimado': 'presupuesto', 
        'ganador': 'ganador', 
        'barrio': 'barrio', 
        'proyecto': 'proyecto'
    }
    
    df_temp = df_temp.rename(columns=mapeo)
    df_temp['año'] = año
    lista_dfs.append(df_temp)

c:\Users\cerru\OneDrive\Documentos\TP FINAL DATOS\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\cerru\OneDrive\Documentos\TP FINAL DATOS\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [225]:
#Confirmamos que todos los años se procesaron bien
print(f"Años cargados en la lista: {[df['año'].unique()[0] for df in lista_dfs]}")

Años cargados en la lista: [np.int64(2017), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [226]:
#Se unifica la lista de todos los años aplicando la función de cateogrías
vilo_master = pd.concat(lista_dfs + [df_2018_final], ignore_index=True)

vilo_master['categoria'] = vilo_master['proyecto'].apply(categorizar_proyecto)
#Además se crea una columna para saber si el proyecto ganó o no
vilo_master['ganador_bin'] = vilo_master['ganador'].apply(lambda x: 1 if str(x).strip().upper() in ['SI', 'GANADOR'] else 0)

#Se guarda el resultado final
vilo_master.to_csv('dataset_final_vilo.csv', index=False, encoding='utf-8-sig')
print("¡Todo unificado con éxito!")

¡Todo unificado con éxito!


In [227]:
#Muestra la lista final de columnas para asegurar que el mapeo funcionó
print(vilo_master.columns.tolist())

['votos', 'orden en la votación', 'ganador', 'barrio', 'proyecto', 'presupuesto', 'suma presupuesto proyectos ganadores', 'año', 'nº de proyecto en la boleta del barrio', 'cantidad de votos obtenidos por el proyecto', 'orden en la votación seguún cantidad de votos obtenidos', 'observaciones', 'suma presupuestos proyectos ganadores por barrio', 'nro. proyecto en el barrio', 'orden en la votación según cantidad de votos', 'descripción', 'n° proyecto en el barrio', 'disponible para el barrio en 2023', 'n° proy en el barrio', 'num barrio', 'disponible para el barrio en 2024', 'disponible para el barrio en 2025', 'disponible para el barrio en 2026', 'categoria', 'ganador_bin']


In [228]:
#Hacemos la copia
df_limpio = vilo_master.copy()

#Unificamos columnas de votos
columnas_votos = ['cantidad de votos obtenidos por el proyecto', 'cantidad de votos obtenidos']
for col in columnas_votos:
    if col in df_limpio.columns:
        df_limpio['votos'] = df_limpio['votos'].fillna(df_limpio[col])

#Unificamos descripción
if 'descripción' in df_limpio.columns:
    df_limpio['proyecto'] = df_limpio['proyecto'].fillna(df_limpio['descripción'])

#Rescatamos ganadores de observaciones
if 'observaciones' in df_limpio.columns:
    mask = df_limpio['observaciones'].str.contains('GANADOR', na=False, case=False)
    df_limpio.loc[mask, 'ganador_bin'] = 1

#Selección de columnas finales
columnas_finales = [
    'año', 
    'barrio', 
    'proyecto', 
    'categoria', 
    'votos', 
    'presupuesto', 
    'ganador_bin'
]
dataset_tif = df_limpio[columnas_finales].copy()

#Limpieza de filas y conversión a ENTEROS (int)
dataset_tif = dataset_tif.dropna(subset=['proyecto', 'barrio'])

dataset_tif['votos'] = pd.to_numeric(dataset_tif['votos'], errors='coerce').fillna(0).astype(int)
dataset_tif['presupuesto'] = pd.to_numeric(dataset_tif['presupuesto'], errors='coerce').fillna(0).astype(int)

dataset_tif.to_csv('dataset_final_vilo.csv', index=False, sep=';', encoding='utf-8-sig')

with open('dataset_final_vilo.csv', 'r+', encoding='utf-8-sig') as f:
    content = f.read()
    f.seek(0, 0)
    f.write('sep=;\n' + content)

print("¡Dataset pulido y guardado con éxito!")
print(f"Total de registros: {len(dataset_tif)}")
dataset_tif.head()


¡Dataset pulido y guardado con éxito!
Total de registros: 1481


,año,barrio,proyecto,categoria,votos,presupuesto,ganador_bin
0,2017,Villa Martelli,BOMBEROS VOLUNTARIOS DE VICENTE LÓPEZ - 25 equ...,Bomberos Voluntarios,959,978500,1
1,2017,Villa Martelli,CAMPAÑA DE CONCIENTIZACIÓN SOBRE EL BULLYING -...,Educación,506,193892,1
2,2017,Villa Martelli,FUNDACION CAMINO - Remodelación interior y fac...,Social,487,1500000,1
3,2017,Villa Martelli,ESCUELA SECUNDARIA Nº 2 (provincial) - Reparac...,Educación,482,1200000,1
4,2017,Villa Martelli,BIBLIOTECA FROILÁN GONZÁLEZ - Compra de mobili...,Educación,446,132000,1


In [229]:
print(dataset_tif['categoria'].value_counts())

categoria
Educación                   454
Social                      267
Deporte                     146
Movilidad                   131
Otros/Varios                 97
Medio Ambiente               77
Seguridad                    69
Cultura                      68
Espacio Público              65
Reparaciones/Obra            56
Bomberos Voluntarios         17
Salud                        12
Instituciones religiosas     11
Cruz Roja Argentina           8
Malvinas                      3
Name: count, dtype: int64
